# 01 — Exploratory Data Analysis & Data Quality

Goal: profile every Olist table, flag data-quality issues, and persist cleaned snapshots to `data/processed/` for downstream notebooks.

Inputs: `data/raw/*.csv`  
Outputs: `data/processed/<table>.parquet`


In [ ]:
from __future__ import annotations

import sys

import pandas as pd

# Make `src/` importable when running from the notebooks/ folder.
sys.path.append('..')

from src.data_loader import load_all_raw, save_processed  # noqa: E402

pd.set_option('display.max_columns', 60)

## Load every available raw table

In [ ]:
tables = load_all_raw()
{name: df.shape for name, df in tables.items()}

## Schema + dtype audit
Confirm dtypes are sane: IDs as strings, money as float, timestamps parsed.

In [ ]:
for name, df in tables.items():
    print(f'\n=== {name} ({df.shape[0]:,} x {df.shape[1]}) ===')
    print(df.dtypes)

## Missingness & duplicates
Flag columns with >5% nulls and any fully duplicated rows.

In [ ]:
def quality_report(df: pd.DataFrame) -> pd.DataFrame:
    null_pct = (df.isna().mean() * 100).round(2)
    return pd.DataFrame({
        'dtype': df.dtypes,
        'null_pct': null_pct,
        'n_unique': df.nunique(),
        'n_duplicates': df.duplicated().sum(),
    })

for name, df in tables.items():
    print(f'\n=== {name} ===')
    display(quality_report(df))

## Save cleaned snapshots
Persist each table to parquet so notebooks 02/03 skip the CSV re-parse.

In [ ]:
for name, df in tables.items():
    path = save_processed(df, name)
    print(f'wrote {path}')